<a href="https://colab.research.google.com/github/e23136/E23136/blob/main/Assignment%2004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [1]:
import plotly.express as px
import plotly.graph_objects as go

class PlottingMethods:
    """
    Reusable Plotly plotting utilities.
    Returns HTML-renderable Plotly figures.
    """

    @staticmethod
    def histogram(df, column):
        fig = px.histogram(
            df,
            x=column,
            title=f"Histogram - {column}"
        )
        return fig

    @staticmethod
    def bar(df, x, y=None, color=None):
        fig = px.bar(
            df,
            x=x,
            y=y,
            color=color,
            title=f"Bar Plot: {x}"
        )
        return fig

    @staticmethod
    def pie(df, names):
        fig = px.pie(
            df,
            names=names,
            title=f"Pie Chart: {names}"
        )
        return fig

    @staticmethod
    def frequency_bar(series):

        counts = series.value_counts(dropna=False)

        perc = round(
            counts / len(series) * 100,
            2
        )

        fig = go.Figure()

        fig.add_trace(
            go.Bar(
                x=counts.index.astype(str),
                y=counts.values,
                text=[f"{p}%" for p in perc],
                textposition='outside'
            )
        )

        fig.update_layout(
            title=f"Frequency Distribution: {series.name}",
            xaxis_title=series.name,
            yaxis_title="Count"
        )

        return fig

    @staticmethod
    def to_html(fig):
        return fig.to_html(full_html=False)
import pandas as pd
import numpy as np

from google.colab import files

from scipy.stats import (
    pearsonr,
    chi2_contingency,
    pointbiserialr,
    f_oneway
)

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder
)

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

import statsmodels.api as sm
class DataInspector:

    GARBAGE_VALUES = [
        '?',
        'n/a',
        'N/A',
        'null',
        'NULL',
        '',
        ' '
    ]

    def __init__(self):
        self.df = None

    #################################################
    # DATA INGESTION
    #################################################

    def upload_data(self):
        """
        Upload CSV from Google Colab.
        """

        uploaded = files.upload()

        file_name = next(iter(uploaded))

        self.df = pd.read_csv(
            file_name,
            na_values=self.GARBAGE_VALUES
        )

        self.auto_type_correction()

        print("Loaded:", file_name)
        print(self.df.shape)

        return self.df

    def load_dataframe(self, df):
        """
        Alternative loader.
        """

        self.df = df.copy()

        self.auto_type_correction()

    def auto_type_correction(self):
        """
        Converts object columns to numeric if possible.
        """

        if self.df is None:
            return

        for col in self.df.columns:

            if self.df[col].dtype == "object":

                converted = pd.to_numeric(
                    self.df[col],
                    errors='coerce'
                )

                if converted.notna().sum() > 0:
                    self.df[col] = converted

    #################################################
    # SUMMARY
    #################################################

    def summary(self):

        if self.df is None:
            print("No data loaded")
            return

        numeric_cols = self.df.select_dtypes(
            include=np.number
        ).columns.tolist()

        categorical_cols = self.df.select_dtypes(
            exclude=np.number
        ).columns.tolist()

        print("=" * 50)
        print("Rows:", self.df.shape[0])
        print("Columns:", self.df.shape[1])
        print("=" * 50)

        print("\nNumeric Columns:")
        print(numeric_cols)

        print("\nCategorical Columns:")
        print(categorical_cols)

        print("\nFirst 20 Rows:")
        display(self.df.head(20))

    #################################################
    # MISSING VALUES
    #################################################

    def handle_missing_values(
            self,
            strategy='mean',
            constant_value=None):

        if self.df is None:
            return

        for col in self.df.columns:

            if self.df[col].isna().sum() == 0:
                continue

            if strategy == 'mean':

                if pd.api.types.is_numeric_dtype(
                        self.df[col]):

                    self.df[col].fillna(
                        self.df[col].mean(),
                        inplace=True
                    )

            elif strategy == 'median':

                if pd.api.types.is_numeric_dtype(
                        self.df[col]):

                    self.df[col].fillna(
                        self.df[col].median(),
                        inplace=True
                    )

            elif strategy == 'mode':

                self.df[col].fillna(
                    self.df[col].mode()[0],
                    inplace=True
                )

            elif strategy == 'constant':

                self.df[col].fillna(
                    constant_value,
                    inplace=True
                )

        return self.df

    #################################################
    # DUPLICATES
    #################################################

    def remove_duplicates(self):

        before = len(self.df)

        self.df.drop_duplicates(inplace=True)

        after = len(self.df)

        print(
            f"Removed {before-after} duplicates"
        )

    #################################################
    # OUTLIERS
    #################################################

    def handle_outliers(
            self,
            column,
            action='flag'):

        q1 = self.df[column].quantile(0.25)
        q3 = self.df[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (
                (self.df[column] < lower)
                |
                (self.df[column] > upper)
        )

        if action == 'flag':

            return self.df[mask]

        elif action == 'remove':

            self.df = self.df[~mask]

            return self.df

    #################################################
    # DELETE ROWS / COLUMNS
    #################################################

    def delete_rows(self):

        rows = input(
            "Enter row indexes (comma separated): "
        )

        rows = [int(x.strip())
                for x in rows.split(',')]

        self.df.drop(
            index=rows,
            inplace=True
        )

    def delete_columns(self):

        cols = input(
            "Enter columns (comma separated): "
        )

        cols = [x.strip()
                for x in cols.split(',')]

        self.df.drop(
            columns=cols,
            inplace=True
        )

    #################################################
    # NORMALIZATION
    #################################################

    def extract_normalized_numeric_data(
            self,
            method='standard'):

        numeric = self.df.select_dtypes(
            include=np.number
        )

        if numeric.empty:
            return pd.DataFrame()

        if method == 'minmax':
            scaler = MinMaxScaler()

        elif method == 'robust':
            scaler = RobustScaler()

        else:
            scaler = StandardScaler()

        scaled = scaler.fit_transform(numeric)

        return pd.DataFrame(
            scaled,
            columns=numeric.columns
        )

    def extract_normalized_categorical_data(
            self,
            method='onehot'):

        cat = self.df.select_dtypes(
            exclude=np.number
        )

        if cat.empty:
            return pd.DataFrame()

        if method == 'onehot':

            encoder = OneHotEncoder(
                sparse_output=False,
                handle_unknown='ignore'
            )

            data = encoder.fit_transform(cat)

            cols = encoder.get_feature_names_out(
                cat.columns
            )

            return pd.DataFrame(
                data,
                columns=cols
            )

        elif method == 'ordinal':

            encoder = OrdinalEncoder()

            data = encoder.fit_transform(cat)

            return pd.DataFrame(
                data,
                columns=cat.columns
            )

        elif method == 'uniform':

            encoder = OrdinalEncoder()

            data = encoder.fit_transform(cat)

            scaler = MinMaxScaler()

            data = scaler.fit_transform(data)

            return pd.DataFrame(
                data,
                columns=cat.columns
            )

    def create_modeling_dataset(
            self,
            cat_method='onehot'):

        numeric = self.df.select_dtypes(
            include=np.number
        )

        cat = self.extract_normalized_categorical_data(
            cat_method
        )

        return pd.concat(
            [numeric.reset_index(drop=True),
             cat.reset_index(drop=True)],
            axis=1
        )

    #################################################
    # VISUALIZATION
    #################################################

    def plot_numeric_distribution(
            self,
            column):

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                "Violin",
                "Scatter",
                "Histogram"
            ]
        )

        fig.add_trace(
            go.Violin(
                y=self.df[column],
                orientation='v'
            ),
            row=1,
            col=1
        )

        fig.add_trace(
            go.Scatter(
                x=self.df.index,
                y=self.df[column],
                mode='markers'
            ),
            row=1,
            col=2
        )

        fig.add_trace(
            go.Histogram(
                x=self.df[column]
            ),
            row=1,
            col=3
        )

        fig.update_layout(
            height=500,
            width=1200,
            title=column
        )

        fig.show()

    def plot_relationship(
            self,
            col1,
            col2):

        is_num1 = pd.api.types.is_numeric_dtype(
            self.df[col1]
        )

        is_num2 = pd.api.types.is_numeric_dtype(
            self.df[col2]
        )

        if is_num1 and is_num2:

            fig = px.scatter(
                self.df,
                x=col1,
                y=col2,
                trendline='ols'
            )

        elif not is_num1 and is_num2:

            fig = px.box(
                self.df,
                x=col1,
                y=col2,
                points='all'
            )

        elif is_num1 and not is_num2:

            fig = px.box(
                self.df,
                x=col2,
                y=col1,
                points='all'
            )

        else:

            temp = pd.crosstab(
                self.df[col1],
                self.df[col2]
            )

            fig = px.bar(
                temp,
                barmode='group'
            )

        fig.show()
    def cramers_v(self, x, y):

        table = pd.crosstab(x, y)

        chi2 = chi2_contingency(table)[0]

        n = table.sum().sum()

        phi2 = chi2 / n

        r, k = table.shape

        return np.sqrt(
            phi2 /
            min(k - 1, r - 1)
        )

    def eta_squared(self,
                    numeric,
                    categorical):

        groups = []

        for cat in categorical.unique():

            groups.append(
                numeric[
                    categorical == cat
                ]
            )

        f, p = f_oneway(*groups)

        ss_between = sum(
            len(g)*(g.mean()-numeric.mean())**2
            for g in groups
        )

        ss_total = (
            (numeric - numeric.mean())**2
        ).sum()

        return np.sqrt(
            ss_between / ss_total
        )

    def plot_all_associations_heatmap(self):

        cols = self.df.columns

        assoc = pd.DataFrame(
            np.zeros(
                (len(cols), len(cols))
            ),
            index=cols,
            columns=cols
        )

        for c1 in cols:

            for c2 in cols:

                if c1 == c2:
                    assoc.loc[c1, c2] = 1
                    continue

                n1 = pd.api.types.is_numeric_dtype(
                    self.df[c1]
                )

                n2 = pd.api.types.is_numeric_dtype(
                    self.df[c2]
                )

                try:

                    if n1 and n2:

                        assoc.loc[c1, c2] = abs(
                            self.df[c1].corr(
                                self.df[c2]
                            )
                        )

                    elif not n1 and not n2:

                        assoc.loc[c1, c2] = self.cramers_v(
                            self.df[c1],
                            self.df[c2]
                        )

                    else:

                        if n1:

                            assoc.loc[c1, c2] = self.eta_squared(
                                self.df[c1],
                                self.df[c2]
                            )

                        else:

                            assoc.loc[c1, c2] = self.eta_squared(
                                self.df[c2],
                                self.df[c1]
                            )

                except:
                    assoc.loc[c1, c2] = np.nan

        fig = px.imshow(
            assoc,
            text_auto=".2f",
            color_continuous_scale='RdBu_r',
            title='Unified Association Heatmap'
        )

        fig.show()

        return assoc


In [2]:
import seaborn as sns

titanic = sns.load_dataset("titanic")

inspector = DataInspector()

inspector.load_dataframe(titanic)

# Summary
inspector.summary()

# Missing values
inspector.handle_missing_values(
    strategy='mode'
)

# Remove duplicates
inspector.remove_duplicates()

# Numeric scaling
scaled_numeric = (
    inspector.extract_normalized_numeric_data(
        method='standard'
    )
)

print(scaled_numeric.head())

# Encoded categories
encoded_cat = (
    inspector.extract_normalized_categorical_data(
        method='onehot'
    )
)

print(encoded_cat.head())

# Combined dataset
model_df = (
    inspector.create_modeling_dataset(
        cat_method='onehot'
    )
)

print(model_df.shape)

# Distribution
inspector.plot_numeric_distribution(
    "age"
)

# Relationship
inspector.plot_relationship(
    "sex",
    "fare"
)

# Heatmap
inspector.plot_all_associations_heatmap()

Rows: 891
Columns: 15

Numeric Columns:
['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']

Categorical Columns:
['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

First 20 Rows:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


/tmp/ipykernel_3938/2335595690.py:237: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[col].fillna(


Removed 113 duplicates
   survived    pclass       age     sibsp     parch      fare
0 -0.842550  0.889003 -0.509883  0.479881 -0.499547 -0.528481
1  1.186874 -1.451587  0.644663  0.479881 -0.499547  0.696162
2  1.186874  0.889003 -0.221246 -0.531901 -0.499547 -0.515571
3  1.186874 -1.451587  0.428185  0.479881 -0.499547  0.348404
4 -0.842550  0.889003  0.428185 -0.531901 -0.499547 -0.513180
   sex_female  sex_male  embarked_C  embarked_Q  embarked_S  class_First  \
0         0.0       1.0         0.0         0.0         1.0          0.0   
1         1.0       0.0         1.0         0.0         0.0          1.0   
2         1.0       0.0         0.0         0.0         1.0          0.0   
3         1.0       0.0         0.0         0.0         1.0          1.0   
4         0.0       1.0         0.0         0.0         1.0          0.0   

   class_Second  class_Third  who_child  who_man  ...  deck_E  deck_F  deck_G  \
0           0.0          1.0        0.0      1.0  ...     0.0     0

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
survived,1.000000,0.331637,0.513353,0.074619,0.039070,0.067344,0.244871,0.168766,0.335542,0.534956,0.525873,0.286209,0.168766,1.000000,0.174188
pclass,0.331637,1.000000,0.113083,0.343283,0.089328,0.042398,0.549330,0.305595,1.000000,0.204883,0.063964,0.585939,0.305595,0.331637,0.109290
sex,0.513353,0.113083,1.000000,0.090755,0.096538,0.234742,0.167447,0.097824,0.125554,0.942249,0.898506,0.184334,0.097824,0.510661,0.278831
age,0.074619,0.343283,0.090755,1.000000,0.274311,0.170766,0.095800,0.067555,0.344364,0.562449,0.265823,0.273310,0.067555,0.074619,0.177980
sibsp,0.039070,0.089328,0.096538,0.274311,1.000000,0.380810,0.134115,0.065012,0.095661,0.434749,0.273191,0.089413,0.065012,0.039070,0.608464
parch,0.067344,0.042398,0.234742,0.170766,0.380810,1.000000,0.190638,0.075387,0.043454,0.414465,0.345204,0.107093,0.075387,0.067344,0.571453
fare,0.244871,0.549330,0.167447,0.095800,0.134115,0.190638,1.000000,0.284463,0.588782,0.183474,0.162977,0.405477,0.284463,0.244871,0.245681
embarked,0.168766,0.305595,0.097824,0.067555,0.065012,0.075387,0.284463,1.000000,0.256151,0.068730,0.084232,0.172569,1.000000,0.168766,0.115087
class,0.335542,1.000000,0.125554,0.344364,0.095661,0.043454,0.588782,0.256151,1.000000,0.152087,0.090232,0.460780,0.256151,0.335542,0.112283
who,0.534956,0.204883,0.942249,0.562449,0.434749,0.414465,0.183474,0.068730,0.152087,1.000000,1.000000,0.168402,0.068730,0.534956,0.437843
